# 🤖 Chat with your from-scratch language model

This notebook downloads **your own trained model** (weights trained from random
initialization — no OpenAI/Claude/Gemini, no pretrained weights, no external AI API)
and lets you chat with it right here.

**How to use (phone friendly):**
1. Tap the **▶ play button** on Step 1 below and wait for `✅ ready` (~30 seconds).
2. Tap ▶ on Step 2. A text box appears at the bottom of the cell.
3. Type a message, press **Enter** on your keyboard. Type `quit` to stop.

*It knows only its tiny training set: trains, cats, dogs, rain, the sun, the moon,
books, rivers, bread, music, birds, snow, trees, boats, clocks, greetings — plus
Alice in Wonderland text. Anything else → rambling. That is honest 3.4M-parameter behavior.*

In [ ]:
#@title Step 1 — press ▶ to download and load the model { display-mode: "form" }
import os, sys

REPO = "https://github.com/debzitsu-ship-it/Project-lmarena.git"
BRANCH = "arena/01a0011c-project-lmarena"

if not os.path.exists("Project-lmarena"):
    !git clone -q -b {BRANCH} {REPO}
%cd -q /content/Project-lmarena
sys.path.insert(0, "/content/Project-lmarena")

import torch
from my_ai.training.trainer import load_checkpoint, pick_device
from my_ai.tokenizer.tokenizer import load_tokenizer, EOS, USER_TOK, ASSISTANT_TOK
from my_ai.inference.generate import generate
from my_ai.chat.cli import build_prompt_ids

device = pick_device()
model, ckpt = load_checkpoint("my_ai/checkpoints/latest_release.pt", device=device)
model.eval()
tokenizer = load_tokenizer("my_ai/checkpoints/tokenizer.json")

print(f"✅ ready — {model.num_parameters():,} parameters | device: {device} | trained to step {ckpt.get('step')}")
print("Now run Step 2 below to start chatting.")

In [ ]:
#@title Step 2 — press ▶ to chat (type in the box, Enter to send, 'quit' to stop) { display-mode: "form" }
history = []
facts = []
print("💬 Chat started. Type your message and press Enter. Commands: 'remember: <fact>', 'facts', 'reset', 'quit'\n")

while True:
    try:
        msg = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not msg:
        continue
    low = msg.lower()
    if low in ("quit", "exit", "/quit"):
        print("👋 bye! Run this cell again to restart the chat.")
        break
    if low == "reset":
        history = []
        print("[context cleared]\n")
        continue
    if low == "facts":
        print("[facts]", facts if facts else "(none)", "\n")
        continue
    if low.startswith("remember:"):
        facts.append(msg.split(":", 1)[1].strip())
        print("[fact stored — in this notebook's memory, not in the model's weights]\n")
        continue

    ids = build_prompt_ids(tokenizer, facts, history, msg, model.cfg.context_length)
    out = generate(model, ids, max_new_tokens=200, temperature=0.7, top_k=40,
                   top_p=0.95, repetition_penalty=1.1,
                   stop_tokens={EOS, USER_TOK, ASSISTANT_TOK}, device=device)
    reply = tokenizer.decode(out).strip() or "(only a stop token)"
    print("AI:", reply, "\n")
    history.append({"role": "user", "text": msg})
    history.append({"role": "assistant", "text": reply})